# ValleyFever-NewsCast

## 3. Time Lag Processing

## Environment Setup

Run the cell below to set up the environment for either Google Colab or local execution:

In [1]:
import os
import sys

# Check if running in Google Colab
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab")

    # Clone repository if in Colab
    if not os.path.exists('/content/ValleyFever-NewsCast/'):
        !git clone https://github.com/Adrian1840/ValleyFever-NewsCast
    os.chdir('/content/ValleyFever-NewsCast')

except ImportError:
    IN_COLAB = False
    print("Running locally")

# Add src directory to Python path
if 'src' not in sys.path:
    sys.path.append('src')

print(f"Current working directory: {os.getcwd()}")

Running in Google Colab
Cloning into 'ValleyFever-NewsCast'...
remote: Enumerating objects: 558, done.
remote: Counting objects: 100% (190/190), done.
remote: Compressing objects: 100% (185/185), done.
remote: Total 558 (delta 101), reused 1 (delta 1), pack-reused 368 (from 3)
Receiving objects: 100% (558/558), 3.87 MiB | 5.27 MiB/s, done.
Resolving deltas: 100% (239/239), done.
Current working directory: /content/ValleyFever-NewsCast


## Import Libraries and Data Loading

In [2]:
  # Basic Packages
  import numpy as np
  import pandas as pd
  from plotting import plot_combined_vf_styled, go

### News Features Data

In [3]:
news_feat_pth = "/content/ValleyFever-NewsCast/data/processed/news_features_unlagged.csv"
news_features_unlagged = pd.read_csv(news_feat_pth)
news_features_unlagged.head(5)

,Year-Month,Num_Articles,cv_mentions_rate,risk_mentions_rate
0,2008-07,0,0.0,0.0
1,2008-08,1,0.0,0.0
2,2008-09,2,0.0,0.0
3,2008-10,0,0.0,0.0
4,2008-11,2,0.0,0.0


### External Data

In [4]:
fresno_agg_pth = "/content/ValleyFever-NewsCast/data/external/Fresno_Aggregate.csv"
kern_agg_pth = "/content/ValleyFever-NewsCast/data/external/Kern_Aggregate.csv"


fresno_agg = pd.read_csv(fresno_agg_pth)
fresno_agg["County"]= "Fresno" #adds county col

kern_agg = pd.read_csv(kern_agg_pth)
kern_agg["County"] = "Kern"

In [5]:
fresno_agg.head(2)

,Year-Month,VFRate,FIRE_Acres_Burned,PRECIP,WIND_EventCount,WIND_AvgMPH,WIND_RunMiles,AQI_PM25,AQI_PM10,EARTHQUAKE_Total,PESTICIDE_Total,County
0,2008-10,6.156349,163.91,0.18,0.0,3.667742,87.825806,70.0,53.0,0,23.056051,Fresno
1,2008-11,3.407979,17.30,1.49,0.0,3.106667,74.490000,95.5,38.5,0,0.519323,Fresno


## Time-Lagging Variables

In [6]:
cols_to_lag = [
    "Num_Articles",
    "cv_mentions_rate",
    "risk_mentions_rate"
]

for lag in [3]:
    for col in cols_to_lag:
        news_features_unlagged[f"{col}_lag{lag}"] = news_features_unlagged[col].shift(lag)

In [8]:
news_features_unlagged

,Year-Month,Num_Articles,cv_mentions_rate,risk_mentions_rate,Num_Articles_lag3,cv_mentions_rate_lag3,risk_mentions_rate_lag3
0,2008-07,0,0.000000,0.000000,NaN,NaN,NaN
1,2008-08,1,0.000000,0.000000,NaN,NaN,NaN
2,2008-09,2,0.000000,0.000000,NaN,NaN,NaN
3,2008-10,0,0.000000,0.000000,0.0,0.000000,0.000000
4,2008-11,2,0.000000,0.000000,1.0,0.000000,0.000000
...,...,...,...,...,...,...,...
85,2015-08,11,0.166667,0.166667,13.0,0.000000,0.000000
86,2015-09,10,0.000000,0.000000,9.0,2.333333,2.333333
87,2015-10,12,0.333333,0.333333,7.0,1.333333,1.333333
88,2015-11,11,0.000000,0.000000,11.0,0.166667,0.166667


In [7]:
# months that match VF rates
months = pd.DataFrame({"Year-Month": pd.period_range("2008-10","2015-12",freq="M").astype(str)})

# Merge with aggregated counts and fill NAs with 0
news_features_lag3 = months.merge(news_features_unlagged[["Year-Month","Num_Articles_lag3","cv_mentions_rate_lag3","risk_mentions_rate_lag3"]], on="Year-Month", how="left")

news_features_lag3.head(5)

,Year-Month,Num_Articles_lag3,cv_mentions_rate_lag3,risk_mentions_rate_lag3
0,2008-10,0.0,0.0,0.0
1,2008-11,1.0,0.0,0.0
2,2008-12,2.0,0.0,0.0
3,2009-01,0.0,0.0,0.0
4,2009-02,2.0,0.0,0.0


In [10]:
news_features_lag3.to_csv("data/processed/news_features_lag3.csv", index=False)

## Log-Transforming Case Rates

In [8]:
#Combining Fresno and Kern aggregates
VF_combined = pd.concat([fresno_agg, kern_agg], ignore_index=True) #concat stacks them on top of each other (since same table just county is different)
VF_combined = VF_combined.sort_values(["County", "Year-Month"]) #sorts d.f. by this
VF_combined = VF_combined[["Year-Month", "VFRate","County"]] #getting only the columns we need for final regression model data frame

VF_combined

,Year-Month,VFRate,County
0,2008-10,6.156349,Fresno
1,2008-11,3.407979,Fresno
2,2008-12,6.486154,Fresno
3,2009-01,6.619800,Fresno
4,2009-02,5.751629,Fresno
...,...,...,...
169,2015-08,12.376954,Kern
170,2015-09,15.329255,Kern
171,2015-10,16.918956,Kern
172,2015-11,16.237656,Kern


In [9]:
VF_combined["Log_Rate"] = np.log1p(VF_combined["VFRate"])

VF_combined = VF_combined[
    ["Year-Month", "County", "Log_Rate"]
]

## Adding Season Variable to Monthly Aggregates

In [10]:
#Adding month to model
news_features_lag3["month"] = pd.to_datetime(news_features_lag3["Year-Month"]).dt.month

In [11]:
def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    else:
        return "Fall"

news_features_lag3["Season"] = news_features_lag3["month"].apply(get_season)

In [ ]:
news_features_lag3.head()

,Year-Month,Num_Articles_lag3,cv_mentions_rate_lag3,risk_mentions_rate_lag3,month,Season
0,2008-10,0.0,0.0,0.0,10,Fall
1,2008-11,1.0,0.0,0.0,11,Fall
2,2008-12,2.0,0.0,0.0,12,Winter
3,2009-01,0.0,0.0,0.0,1,Winter
4,2009-02,2.0,0.0,0.0,2,Winter


## Creating Final Model Data Frame

In [12]:
final_model_dataframe = (
    VF_combined.merge(
        news_features_lag3,
        on="Year-Month",
        how="left"
    )
)

# Reorder columns
final_model_dataframe = final_model_dataframe[
    [
        "Log_Rate",
        "Num_Articles_lag3",
        "cv_mentions_rate_lag3",
        "risk_mentions_rate_lag3",
        "County",
        "Season",
        "Year-Month"
    ]
]

final_model_dataframe.head()

,Log_Rate,Num_Articles_lag3,cv_mentions_rate_lag3,risk_mentions_rate_lag3,County,Season,Year-Month
0,1.968000,0.0,0.0,0.0,Fresno,Fall,2008-10
1,1.483416,1.0,0.0,0.0,Fresno,Fall,2008-11
2,2.013055,2.0,0.0,0.0,Fresno,Winter,2008-12
3,2.030750,0.0,0.0,0.0,Fresno,Winter,2009-01
4,1.909784,2.0,0.0,0.0,Fresno,Winter,2009-02


In [ ]:
final_model_dataframe.to_csv("data/processed/final_model_dataframe.csv", index=False)

# Plotting News Media Volume vs Case Rate

Located in `results/`

In [13]:
kern_news_plot = kern_agg.merge(
    news_features_unlagged[["Year-Month", "Num_Articles"]],
    on="Year-Month",
    how="left"
).fillna(0)

fig_kern = plot_combined_vf_styled("Kern", kern_news_plot)

fig_kern.show()